In [ ]:
import dotenv
import pendulum
from sqlalchemy import asc, cast
from sqlalchemy import create_engine, select, BigInteger
from sqlalchemy.orm import sessionmaker
from sqlalchemy import create_engine, select, BigInteger
from config import Settings
from models import MessageSql
from pydantic import BaseModel, Field

class Glitch(BaseModel):
    FromGNodeAlias: str
    Node: str
    Type: str
    Summary: str
    Details: str
    CreatedMs: int
    TypeName: str
    Version: str

house_alias = "fir"
message_type = "glitch"
start_ms = pendulum.datetime(2025, 12, 20, 0, tz='America/New_York').timestamp()*1000
end_ms = pendulum.datetime(2026, 1, 10, 0, tz='America/New_York').timestamp()*1000

stmt = select(MessageSql).filter(
    MessageSql.message_type_name == message_type,
    MessageSql.from_alias == f"hw1.isone.me.versant.keene.{house_alias}.scada",
    MessageSql.message_persisted_ms <= cast(int(end_ms), BigInteger),
    MessageSql.message_persisted_ms >= cast(int(start_ms), BigInteger),
).order_by(asc(MessageSql.message_persisted_ms))

settings = Settings(_env_file=dotenv.find_dotenv())
engine = create_engine(settings.db_url_no_async.get_secret_value())
Session = sessionmaker(bind=engine)
session = Session()
result = session.execute(stmt)
messages = result.scalars().all()

print(f"Found {len(messages)} messages")


In [ ]:
PRINT = False
times = []
online = []

for m in messages:
    glitch = Glitch(**m.payload)
    # print(glitch.Summary)
    # print(glitch.Details)

    if glitch.Summary == 'pico-just-zombied':
        if PRINT: print("Just zombied at", pendulum.from_timestamp(m.message_persisted_ms/1000, tz='America/New_York'))
        times.append(pendulum.from_timestamp(m.message_persisted_ms/1000, tz='America/New_York'))
        online.append(0)
    elif "[tank3] recovered from zombie state" in glitch.Details:
        if PRINT: print("Recovered from zombie state at", pendulum.from_timestamp(m.message_persisted_ms/1000, tz='America/New_York'))
        times.append(pendulum.from_timestamp(m.message_persisted_ms/1000, tz='America/New_York'))
        online.append(1)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 2))
plt.step(times, online, where='post')
plt.yticks([0, 1])
plt.gca().set_yticklabels(['zombied', 'online'])
plt.title(f'Fir tank3 state')
import matplotlib.dates as mdates

# Set the x-axis date format to 'mm/dd hh:00'
date_format = mdates.DateFormatter('%m/%d %H:00')
plt.gca().xaxis.set_major_formatter(date_format)
plt.gcf().autofmt_xdate()

plt.show()

In [ ]:
from pathlib import Path
import sys

import dotenv
import pendulum
from sqlalchemy import asc, cast
from sqlalchemy import create_engine, select, BigInteger
from sqlalchemy.orm import sessionmaker

# Ensure repo root is on sys.path for both script and notebook execution.
# try:
#     REPO_ROOT = Path(__file__).resolve().parents[2]
# except NameError:
#     cwd = Path.cwd().resolve()
#     REPO_ROOT = cwd if (cwd / "visualizer").exists() else cwd.parents[1]

# if str(REPO_ROOT) not in sys.path:
#     sys.path.insert(0, str(REPO_ROOT))

from visualizer.config import Settings
from visualizer.models import MessageSql
import matplotlib.pyplot as plt

house_alias = "spruce"
message_type = "snapshot.spaceheat"
start_ms = pendulum.datetime(2026, 4, 8, 0, 0, 0, tz='America/New_York').timestamp()*1000
end_ms = pendulum.datetime(2026, 4, 9, 0, 0, 0, tz='America/New_York').timestamp()*1000

stmt = select(MessageSql).filter(
    MessageSql.message_type_name == message_type,
    MessageSql.from_alias == f"hw1.isone.me.versant.keene.{house_alias}.scada",
    MessageSql.message_created_ms <= cast(int(end_ms), BigInteger),
    MessageSql.message_created_ms >= cast(int(start_ms), BigInteger),
).order_by(asc(MessageSql.message_persisted_ms))

settings = Settings(_env_file=dotenv.find_dotenv())
engine = create_engine(settings.db_url_no_async.get_secret_value())
Session = sessionmaker(bind=engine)
session = Session()
result = session.execute(stmt)
messages = result.scalars().all()

print(f"Found {len(messages)} messages")

In [ ]:
messages[0].payload

In [ ]:
messages[0].payload.keys()

In [ ]:
m = messages[0].payload
m['LatestReadingList'][0].keys()

In [ ]:
channels = {}

for mess in messages:
    m = mess.payload
    for r in m['LatestReadingList']:
        if r['ChannelName'] not in channels:
            print(r['ChannelName'])
            channels[r['ChannelName']]= {
                'times': [],
                'values': []
            }
        channels[r['ChannelName']]['times'].append(r['ScadaReadTimeUnixMs'])
        channels[r['ChannelName']]['values'].append(r['Value'])

channels.keys()

selected = 'floor-swt'
plt.plot(channels[selected]['times'], channels[selected]['values'])
plt.show()

In [ ]:
'''
zone1-bedrooms-opto-input
zone2-living-rm-opto-input
zone4-garage-opto-input
zone1-bedrooms-gw-temp
zone1-bedrooms-gw-microvolts
zone2-living-rm-gw-temp
zone2-living-rm-gw-microvolts
zone1-bedrooms-floor-temp
zone2-living-rm-floor-temp
zone4-garage-floor-temp
zone1-bedrooms-heat-call
zone2-living-rm-heat-call
zone4-garage-heat-call
'''

In [ ]:
 'pipes1-depth1-device',
 'pipes1-depth2-device',
 'pipes1-depth3-device',

 'floor1-depth1-device',
 'floor1-depth2-device',
 'floor1-depth3-device',

 'zone1-bedrooms-opto-input',
 'zone2-living-rm-opto-input',
 'zone4-garage-opto-input',

 'zone1-bedrooms-floor-temp',
 'zone2-living-rm-floor-temp',
 'zone4-garage-floor-temp'

In [ ]:
from pathlib import Path
import sys

import dotenv
import pendulum
from sqlalchemy import asc, cast
from sqlalchemy import create_engine, select, BigInteger
from sqlalchemy.orm import sessionmaker
from sqlalchemy import create_engine, select, BigInteger

from visualizer.config import Settings
from visualizer.models import MessageSql
import matplotlib.pyplot as plt

house_alias = "spruce"
message_type = "snapshot.spaceheat"
start_ms = pendulum.datetime(2026, 4, 15, 12, 0, 0, tz='America/New_York').timestamp()*1000
end_ms = pendulum.datetime(2026, 4, 15, 13, 0, 0, tz='America/New_York').timestamp()*1000

stmt = select(MessageSql).filter(
    MessageSql.message_type_name == message_type,
    MessageSql.from_alias == f"hw1.isone.me.versant.keene.{house_alias}.scada",
    MessageSql.message_created_ms <= cast(int(end_ms), BigInteger),
    MessageSql.message_created_ms >= cast(int(start_ms), BigInteger),
).order_by(asc(MessageSql.message_persisted_ms))

settings = Settings(_env_file=dotenv.find_dotenv())
engine = create_engine(settings.db_url_no_async.get_secret_value())
Session = sessionmaker(bind=engine)
session = Session()
result = session.execute(stmt)
messages = result.scalars().all()

print(f"Found {len(messages)} messages")

print(messages[0].payload.keys())

buffer-depth1-device
buffer-depth2-device
buffer-depth3-device
buffer-depth1-micro-v
buffer-depth2-micro-v
buffer-depth3-micro-v
dist-flow
dist-swt
dist-rwt
pipes1-depth1-device
pipes1-depth2-device
pipes1-depth1-micro-v
pipes1-depth2-micro-v
pipes1-depth3-micro-v
floor1-depth1-device
floor1-depth2-device
floor1-depth3-device
floor1-depth1-micro-v
floor1-depth2-micro-v
floor1-depth3-micro-v
elt-buffer-top-pwr
elt-buffer-bottom-pwr
elt-store-top-pwr
elt-store-bottom-pwr
dist-pump-pwr
zone1-bedrooms-opto-input
zone2-living-rm-opto-input
zone4-garage-opto-input
zone1-bedrooms-gw-temp
zone1-bedrooms-gw-microvolts
zone2-living-rm-gw-temp
zone2-living-rm-gw-microvolts
zone4-garage-gw-temp
zone4-garage-gw-microvolts
buffer-depth1
buffer-depth2
buffer-depth3
floor-swt
floor-rwt
zone1-bedrooms-floor-temp
zone2-living-rm-floor-temp
zone4-garage-floor-temp
zone1-bedrooms-heat-call
zone2-living-rm-heat-call
zone4-garage-heat-call


NameError: name 'plt' is not defined